In [ ]:
!pip install transformers datasets sentencepiece accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 13.9 MB/s eta 0:00:00


In [ ]:
from google.colab import files
uploaded = files.upload()

filename = list(uploaded.keys())[0]
filename

Saving autodoc_dataset.json to autodoc_dataset.json


'autodoc_dataset.json'

In [ ]:
import json
import pandas as pd

with open(filename, "r") as f:
    data = json.load(f)

rows = []
for item in data:
    note = item["note"]

    # convert dictionary summary into a text target
    summary = item["summary"]
    target = (
        f"Complaint: {summary['Complaint']}\n"
        f"History: {summary['History']}\n"
        f"Medication: {summary['Medication']}\n"
        f"Plan: {summary['Plan']}"
    )

    rows.append({
        "input_text": note,
        "target_text": target
    })

df = pd.DataFrame(rows)
df.head()

,input_text,target_text
0,M patient aged 47 years admitted for Atrial fi...,Complaint: Atrial fibrillation\nHistory: Known...
1,M patient aged 58 years admitted for Backgroun...,Complaint: Background diabetic retinopathy\nHi...
2,M patient aged 79 years admitted for Personal ...,Complaint: Personal history of tobacco use\nHi...
3,M patient aged 52 years admitted for Esophagea...,Complaint: Esophageal varices in diseases clas...
4,F patient aged 82 years admitted for Dehydrati...,Complaint: Dehydration\nHistory: No significan...


In [ ]:
from datasets import Dataset

dataset = Dataset.from_pandas(df)
dataset = dataset.train_test_split(test_size=0.1)

train_ds = dataset["train"]
test_ds = dataset["test"]


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [ ]:
def preprocess(batch):
    model_inputs = tokenizer(
        batch["input_text"],
        max_length=512,
        padding="max_length",
        truncation=True
    )

    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            batch["target_text"],
            max_length=256,
            padding="max_length",
            truncation=True
        )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_ds = train_ds.map(preprocess, batched=True)
test_ds = test_ds.map(preprocess, batched=True)


Map:   0%|          | 0/247 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:4034: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


Map:   0%|          | 0/28 [00:00<?, ? examples/s]

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./autodoc_finetuned",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    weight_decay=0.01,
    num_train_epochs=3,
    fp16=True,  # Colab GPU supports mixed precision
    push_to_hub=False
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    tokenizer=tokenizer
)


/tmp/ipython-input-4180416297.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: candystress6 (candystress6-student) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss
1,No log,nan
2,No log,nan
3,No log,nan


TrainOutput(global_step=372, training_loss=0.0, metrics={'train_runtime': 238.5635, 'train_samples_per_second': 3.106, 'train_steps_per_second': 1.559, 'total_flos': 507405198163968.0, 'train_loss': 0.0, 'epoch': 3.0})

In [ ]:
trainer.save_model("./autodoc_finetuned")
tokenizer.save_pretrained("./autodoc_finetuned")

('./autodoc_finetuned/tokenizer_config.json',
 './autodoc_finetuned/special_tokens_map.json',
 './autodoc_finetuned/spiece.model',
 './autodoc_finetuned/added_tokens.json',
 './autodoc_finetuned/tokenizer.json')